In [ ]:
from typing import Any, Callable, Sequence
from sqlmodel import create_engine, select, Session
from sqlalchemy.engine import Engine
from sqlalchemy import event
from experiment import Model, Result, Celltype, Dataset, Sample, NumericArray
import pandas as pd
import mlflow
import logging
import polars as pl
import nico2_lib as n2l
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
exp_name = "Default"
benchmark_experiment = mlflow.get_experiment_by_name(exp_name)
if not benchmark_experiment:
    raise ValueError(f"Experiment '{exp_name}' not found.")
runs = mlflow.search_runs(experiment_ids=[benchmark_experiment.experiment_id])
if runs.empty:
    raise RuntimeError(f"No runs found in experiment '{exp_name}'.")
last_run_id = runs.sort_values("start_time", ascending=False)["run_id"].iloc[0]
logger.info(f"Using run_id: {last_run_id}")
logger.info("Downloading 'database.db'...")
database_path = mlflow.artifacts.download_artifacts(
    run_id=last_run_id, artifact_path="database.db"
)
logger.info(f"Local path: {database_path}")
sqlite_url = f"sqlite:///{database_path}"
engine = create_engine(sqlite_url, echo=True)
logger.info("Database engine initialized.")

In [ ]:
def plot_embedding(results: Sequence[tuple[Result, Model]], celltype: Celltype, sample: Sample) -> None:
    n_rows = len(results)
    n_cols = max(r.celltype_model_embedding.shape[1] for r, _ in results)
    umap = celltype.umap_embedding

    _, axes = plt.subplots(
        n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5), squeeze=False, dpi=300
    )

    for r_idx, (res, model) in enumerate(results):
        latents = res.celltype_model_embedding
        for c_idx in range(n_cols):
            ax = axes[r_idx, c_idx]
            if c_idx < latents.shape[1]:
                ax.scatter(*umap.T, c=latents[:, c_idx], cmap="plasma", s=5, alpha=0.8)
                ax.set_title(f"Latent {c_idx}")
                ax.set_ylabel(
                    f"{model.name}\n(k={latents.shape[1]})", fontweight="bold"
                )

    ds_name = getattr(celltype.dataset, "name", "Unknown")
    plt.suptitle(
        f"Comparison: {celltype.name} | Dataset: {ds_name} | Sample: {sample.id_of_sample}"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
from itertools import product


with Session(engine) as session:
    datasets = session.exec(select(Dataset)).all()
    for dataset in datasets:
        celltypes = dataset.celltypes
        samples = dataset.samples
        for celltype, sample in product(celltypes, samples):
            results = session.exec(
                select(Result, Model)
                .join(Model)
                .where(Result.celltype == celltype)
                .where(Result.sample == sample)
            ).all()
            plot_embedding(results, celltype, sample)

In [ ]:
def mock_metrics_func(result: Result) -> list[dict[str, Any]]:
    rng = np.random.default_rng()
    return [
        {"metric_name": "pearsonr", "metric_score": rng.normal()},
        {"metric_name": "spearmanr", "metric_score": rng.normal()},
    ]

def results_to_dataframe(
    results: Sequence[Result],
    func: Callable[[Result], list[dict[str, Any]]],
) -> pl.DataFrame:
    rows = []
    for result in results:
        extracted_results = func(result)
        for extracted_result in extracted_results:
            rows.append(
                {
                    **extracted_result,
                    "dataset_name": result.celltype.dataset.name,
                    "celltype": result.celltype.name,
                    "sample_id": result.sample.id_of_sample,
                    "model_name": result.model.name
                }
            )
    return pl.DataFrame(rows)


In [ ]:
with Session(engine) as session:
    results = session.exec(select(Result)).all()
    df = results_to_dataframe(results, mock_metrics_func)

In [ ]:
g = sns.FacetGrid(df, col="metric_name")
g.map_dataframe(sns.boxplot, x="model_name", y="metric_score")